In [1]:
import os

os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/dushyantverma414/DS-Project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="dushyantverma414"
os.environ["MLFLOW_TRACKING_PASSWORD"]="25dab9c8e5112fc3db379fd131b2ba826a9fcb5f"

In [2]:
import os
os.chdir("../")
%pwd

'd:\\Data_analytics_new\\FSDS\\learning\\DS_Project\\DS-Project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [4]:
from src.ds_project.constants import *
from src.ds_project.utils.common import read_yaml, create_directories, save_bin, save_json

In [5]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        #print(self.config.artifacts_root)

        create_directories(self.config.artifacts_root)

    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        create_directories(config.root_dir)

        model_evaluation_config=ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/dushyantverma414/DS-Project.mlflow"
        )
        return model_evaluation_config

In [6]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [9]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config=config

        ### Add any techniques here

    def eval_metrics(self, actual, pred):
        rmse=np.sqrt(mean_squared_error(actual, pred))
        mse=mean_absolute_error(actual, pred)
        r2=r2_score(actual, pred)
        return rmse, mse, r2
    
    def log_into_mlflow(self):
        test_data=pd.read_csv(self.config.test_data_path)
        model=joblib.load(self.config.model_path)

        test_x=test_data.drop([self.config.target_column], axis=1)
        test_y=test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_uri_type_store=urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            ## saving metrics a slocal
            scores = {"rmse":rmse, "mae":mae, "r2":r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            ## model registry does not work with file store
            if tracking_uri_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.save_model(model, "model")

      

In [10]:
try:
    config=ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e


<_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='utf-8'>
{'artifacts_root': ['artifacts'], 'data_ingestion': {'root_dir': ['artifacts/data_ingestion'], 'source_URL': 'https://raw.githubusercontent.com/dushyantverma22/data_set/main/WineQT.csv', 'local_data_file': 'artifacts/data_ingestion/WineQT.csv', 'unzip_dir': 'artifacts/data_ingestion'}, 'data_validation': {'root_dir': ['artifacts/data_validation'], 'unzip_data_dir': 'artifacts/data_ingestion/WineQT.csv', 'STATUS_FILE': 'artifacts/data_validation/status.txt'}, 'data_transformation': {'root_dir': 'artifacts/data_transforamtion', 'data_path': 'artifacts/data_ingestion/WineQT.csv'}, 'model_trainer': {'root_dir': 'artifacts/model_trainer', 'train_file_path': 'artifacts/data_transforamtion/train.csv', 'test_file_path': 'artifacts/data_transforamtion/test.csv', 'model_name': 'model.joblib'}, 'model_evaluation': {'root_dir': 'artifacts/model_evaluation', 'test_data_path': 'artifacts/data_transforamtion/test.csv', 'model_pa

2024/10/22 01:38:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'ElasticNetModel'.
2024/10/22 01:39:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.
2024/10/22 01:39:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run vaunted-panda-842 at: https://dagshub.com/dushyantverma414/DS-Project.mlflow/#/experiments/0/runs/2ba26a848d104b83bb9a922c521f2dea.
2024/10/22 01:39:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/dushyantverma414/DS-Project.mlflow/#/experiments/0.
